# **TikTok Project**
**Course 3 — The Power of Statistics**

**Author:** Ahmad Daniel  
**Date:** May 2026

---

## Project Overview

The TikTok data team has completed EDA and visualization (Courses 1 & 2). The next milestone is **statistical hypothesis testing**.

**Request from stakeholders:**  
Determine whether there is a statistically significant difference in the number of views for TikTok videos posted by **verified accounts** versus **unverified accounts**.

**This notebook will:**
1. Compute descriptive statistics on the claims classification data
2. Conduct a two-sample hypothesis test on `video_view_count` by `verified_status`
3. Interpret and communicate the results

---
# PACE: Plan

## Task 1. Imports and data loading

In [ ]:
# Import packages
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
pd.set_option('display.max_columns', None)

In [ ]:
# Load dataset
data = pd.read_csv('tiktok_dataset.csv')
print(f'Dataset shape: {data.shape}')

---
# PACE: Analyze

## Task 2. Explore the data

### 2a. Examine the data

In [ ]:
# Display first 5 rows
data.head()

In [ ]:
# Get data types and null counts
data.info()

In [ ]:
# Drop null rows (same decision as Course 2)
data = data.dropna()
print(f'Working dataset: {data.shape}')

### 2b. Compute descriptive statistics

In [ ]:
# Overall descriptive statistics
data.describe()

In [ ]:
# Check verified_status distribution
print('Verified status distribution:')
print(data['verified_status'].value_counts())
print(f'\nPercentage breakdown:')
print(data['verified_status'].value_counts(normalize=True).mul(100).round(2))

In [ ]:
# Descriptive statistics for video_view_count by verified_status
data.groupby('verified_status')['video_view_count'].agg(
    count='count',
    mean='mean',
    median='median',
    std='std',
    min='min',
    max='max'
).round(2)

**Observation:** There is a notable difference in mean video view counts between verified and not verified accounts:
- **Verified accounts** mean: ~88,809 views
- **Not verified accounts** mean: ~264,332 views

Not verified accounts appear to have significantly higher mean view counts. However, the standard deviations are very large relative to the means, indicating heavy spread and skew in both groups. A formal hypothesis test is needed to determine whether this difference is statistically significant.

In [ ]:
# Visualize the distribution of video_view_count by verified_status
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
for status, color in [('verified', '#1f77b4'), ('not verified', '#ff7f0e')]:
    subset = data[data['verified_status'] == status]['video_view_count']
    axes[0].hist(subset, bins=50, alpha=0.6, color=color, label=status, edgecolor='white')

axes[0].set_title('Video View Count Distribution by Verified Status', fontsize=11, fontweight='bold')
axes[0].set_xlabel('video_view_count', fontsize=10)
axes[0].set_ylabel('Count', fontsize=10)
axes[0].legend(title='Verified Status')

# Boxplot
sns.boxplot(data=data, x='verified_status', y='video_view_count',
            hue='verified_status',
            palette={'verified': '#1f77b4', 'not verified': '#ff7f0e'},
            ax=axes[1], width=0.5, legend=False,
            flierprops=dict(marker='o', markersize=2, alpha=0.3))
axes[1].set_title('Video View Count Boxplot by Verified Status', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Verified Status', fontsize=10)
axes[1].set_ylabel('video_view_count', fontsize=10)

plt.tight_layout()
plt.show()

---
# PACE: Construct

## Task 3. Hypothesis testing

### 3a. State the hypotheses

**Research question:**  
Is there a statistically significant difference in the mean video view count between verified and not verified TikTok accounts?

**Hypotheses:**
- **H₀ (Null hypothesis):** There is no difference in the mean video view count between verified and not verified accounts. Any observed difference is due to random chance.
- **Hₐ (Alternative hypothesis):** There IS a statistically significant difference in the mean video view count between verified and not verified accounts.

**Significance level:** α = 0.05

**Test choice:** Two-sample t-test (Welch's t-test)  
- Two independent groups: verified vs not verified
- Continuous numeric dependent variable: `video_view_count`
- Welch's variant used because group variances are likely unequal (standard deviations differ greatly)
- Two-tailed test because we are testing for a difference in either direction

### 3b. Check assumptions

In [ ]:
# Separate the two groups
verified = data[data['verified_status'] == 'verified']['video_view_count']
not_verified = data[data['verified_status'] == 'not verified']['video_view_count']

print(f'Verified group size: {len(verified):,}')
print(f'Not verified group size: {len(not_verified):,}')
print(f'\nCentral Limit Theorem applies for both groups (n > 30) ✓')
print(f'Groups are independent ✓')
print(f'Using Welch\'s t-test (does not assume equal variances) ✓')

### 3c. Conduct the two-sample t-test

In [ ]:
# Conduct Welch's two-sample t-test
t_stat, p_value = stats.ttest_ind(verified, not_verified, equal_var=False)

print('=' * 50)
print('TWO-SAMPLE T-TEST RESULTS')
print('=' * 50)
print(f'H₀: Mean view count is equal for both groups')
print(f'Hₐ: Mean view count differs between groups')
print(f'Significance level (α): 0.05')
print(f'\nt-statistic: {t_stat:.4f}')
print(f'p-value:     {p_value:.6f}')
print('=' * 50)

if p_value < 0.05:
    print(f'\nResult: REJECT H₀')
    print(f'The p-value ({p_value:.6f}) is less than α (0.05).')
    print(f'There IS a statistically significant difference in video')
    print(f'view counts between verified and not verified accounts.')
else:
    print(f'\nResult: FAIL TO REJECT H₀')
    print(f'The p-value ({p_value:.6f}) is greater than α (0.05).')

---
# PACE: Execute

## Task 4. Interpret and communicate results

### Statistical interpretation

The two-sample Welch's t-test produced:
- **t-statistic = -25.29** — a large negative value indicating the verified group mean is substantially lower than the not verified group mean
- **p-value ≈ 0.000** — effectively zero, far below the significance level of 0.05

**Decision:** Reject the null hypothesis.

**Conclusion:** There is a statistically significant difference in the mean video view count between verified and not verified TikTok accounts. Not verified accounts have a higher mean view count (~264,332) compared to verified accounts (~88,809).

### Business interpretation

This finding is counterintuitive — we might expect verified (more trusted/prominent) accounts to get more views. However, this result aligns with patterns observed in Course 2:

- Not verified accounts are more likely to post **claims** (unverified information)
- Claim videos generate dramatically higher engagement (views, likes, shares) than opinion videos
- So not verified accounts → more claims → more views (driven by sensational content, not account credibility)

**Important caveat:** Statistical significance does not imply causation. This result tells us that the difference is real (not due to random chance), but not *why* it exists. The relationship between verified status, claim status, and engagement is complex and needs careful interpretation before being built into the model.

### Recommended next steps

1. Investigate the relationship between `verified_status` and `claim_status` — are not verified accounts disproportionately posting claims?
2. Consider `verified_status` as a potential feature in the classification model — but with caution to avoid proxy bias
3. Conduct additional hypothesis tests on other engagement variables (likes, shares, comments) to build a complete picture
4. Move forward to regression analysis (Course 4) to model the relationship between multiple variables simultaneously